# Notebook 03 — Evaluate Fine-tuned Model

Loads the fine-tuned model from HuggingFace and evaluates it against  
the held-out validation set produced by notebook 01.

In [ ]:
%%capture
!pip install transformers datasets

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import Optional, Tuple
from pathlib import Path
from tqdm import tqdm

from datasets import load_dataset
from transformers import AutoConfig, Wav2Vec2FeatureExtractor, Wav2Vec2PreTrainedModel, Wav2Vec2Model
from transformers.file_utils import ModelOutput
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

In [ ]:
data_path          = Path('/content/drive/MyDrive/CS7357/Project/data')
model_name_or_path = 'm0nkspade/wav2vec2-large-xls-r-300m-dm32'
base_model_path    = 'facebook/wav2vec2-large-xlsr-53'  # fallback for feature extractor
input_col          = 'path'
output_col         = 'label'

## 1. Load Validation Set

In [ ]:
dataset   = load_dataset('csv', data_files={'valid': str(data_path / 'valid_dm.csv')}, delimiter='\t')
test_data = dataset['valid']

label_list  = sorted(test_data.unique(output_col))
num_classes = len(label_list)
label2id    = {label: i for i, label in enumerate(label_list)}
id2label    = {i: label for i, label in enumerate(label_list)}

print(f'Validation samples: {len(test_data)}')
print(f'Classes ({num_classes}): {label_list}')

## 2. Model Architecture

In [ ]:
@dataclass
class SpeechClassifierOutput(ModelOutput):
    loss:          Optional[torch.FloatTensor]         = None
    logits:        torch.FloatTensor                   = None
    hidden_states: Optional[Tuple[torch.FloatTensor]]  = None
    attentions:    Optional[Tuple[torch.FloatTensor]]  = None


class Wav2Vec2ClassificationHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense    = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout  = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, x, **kwargs):
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class Wav2Vec2ForSpeechClassification(Wav2Vec2PreTrainedModel):
    all_tied_weights_keys = {}

    def __init__(self, config):
        super().__init__(config)
        self.num_labels   = config.num_labels
        self.pooling_mode = config.pooling_mode
        self.wav2vec2     = Wav2Vec2Model(config)
        self.classifier   = Wav2Vec2ClassificationHead(config)
        self.init_weights()

    def _pool(self, hidden_states):
        if self.pooling_mode == 'mean': return hidden_states.mean(dim=1)
        if self.pooling_mode == 'max':  return hidden_states.max(dim=1)[0]
        if self.pooling_mode == 'sum':  return hidden_states.sum(dim=1)
        raise ValueError(f'Unknown pooling mode: {self.pooling_mode}')

    def forward(self, input_values, attention_mask=None, output_attentions=None,
                output_hidden_states=None, return_dict=None, labels=None):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask,
                                output_attentions=output_attentions,
                                output_hidden_states=output_hidden_states,
                                return_dict=return_dict)
        logits = self.classifier(self._pool(outputs[0]))
        loss   = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))
        if not return_dict:
            return ((loss, logits) + outputs[2:]) if loss is not None else (logits,) + outputs[2:]
        return SpeechClassifierOutput(loss=loss, logits=logits,
                                      hidden_states=outputs.hidden_states,
                                      attentions=outputs.attentions)

## 3. Load Model

In [ ]:
# Transformers inspects __main__.__file__ when loading a custom model class in a notebook
if not hasattr(sys.modules['__main__'], '__file__'):
    with open('notebook_source.py', 'w') as f:
        f.write('')
    sys.modules['__main__'].__file__ = os.path.abspath('notebook_source.py')

config = AutoConfig.from_pretrained(model_name_or_path)
config.num_labels = num_classes
config.label2id   = label2id
config.id2label   = id2label
config.pooling_mode = 'mean'

# Feature extractor — fall back to base model if not pushed with fine-tuned weights
try:
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name_or_path)
except OSError:
    print(f'Feature extractor not found in {model_name_or_path}, using {base_model_path}')
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(base_model_path)

model = Wav2Vec2ForSpeechClassification.from_pretrained(model_name_or_path, config=config).to(device)
model.eval()
print(f'Model loaded on {device}.')

## 4. Run Inference

In [ ]:
dem_id = label_list.index('dementia')

def speech_to_array(batch):
    """Load and resample audio to 16 kHz. Uses the full clip for evaluation."""
    speech, sr = torchaudio.load(batch['path'])
    speech     = torchaudio.transforms.Resample(sr, 16000)(speech)[0].numpy()
    batch['speech'] = speech
    return batch


def predict(batch):
    features = feature_extractor(
        batch['speech'],
        sampling_rate=feature_extractor.sampling_rate,
        return_tensors='pt',
        padding=True,
    )
    input_values = features.input_values.to(device)
    with torch.no_grad():
        logits = model(input_values).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    batch['prob_dementia'] = probs[:, dem_id].tolist()
    batch['predicted']     = probs[:, dem_id].tolist()  # raw probs; threshold applied below
    return batch


test_data = test_data.map(speech_to_array)
result    = test_data.map(predict, batched=True, batch_size=8)
print('Inference complete.')

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prob_dem   = np.array(result['prob_dementia'])
y_true_arr = np.array([label2id[l] for l in result['label']])

print(f"{'Threshold':>10}  {'Dem Prec':>10}  {'Dem Recall':>10}  {'Dem F1':>8}  {'Accuracy':>10}")
print('-' * 62)
for thresh in np.arange(0.20, 0.55, 0.05):
    preds        = (prob_dem >= thresh).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true_arr, preds, labels=[dem_id], average=None)
    acc          = (preds == y_true_arr).mean()
    print(f"{thresh:>10.2f}  {p[0]:>10.3f}  {r[0]:>10.3f}  {f1[0]:>8.3f}  {acc:>10.3f}")

# ── Set your chosen threshold here ──────────────────────────────────────────
DEMENTIA_THRESHOLD = 0.35
print(f'\nUsing threshold: {DEMENTIA_THRESHOLD}')

## 5. Results

In [ ]:
prob_dem      = np.array(result['prob_dementia'])
y_true        = [label2id[l] for l in result['label']]
y_pred        = (prob_dem >= DEMENTIA_THRESHOLD).astype(int).tolist()
y_true_labels = result['label']
y_pred_labels = [id2label[i] for i in y_pred]

print(f'Classification report (dementia threshold = {DEMENTIA_THRESHOLD}):')
print(classification_report(y_true, y_pred, target_names=label_list))

In [ ]:
cm    = confusion_matrix(y_true_labels, y_pred_labels, labels=label_list)
cm_df = pd.DataFrame(cm, index=label_list, columns=label_list)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix (dementia threshold = {DEMENTIA_THRESHOLD})')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()